In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../')
from utils.MultiLabelPredictor import MultilabelPredictor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.base import clone
import numpy as np
import matplotlib.pyplot as plt

/home/massimo/Documents/stage/signature_inference/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def compare_models(r2_1, mse_1, r2_2, mse_2, thresholds=[0.5, 0.8]):
    r2_1, mse_1 = np.array(r2_1), np.array(mse_1)
    r2_2, mse_2 = np.array(r2_2), np.array(mse_2)

    def summary_stats(metric):
        return (np.nanmean(metric), np.nanmedian(metric), np.nanmax(metric), np.nanmin(metric))

    r2_1_stats = summary_stats(r2_1)
    r2_2_stats = summary_stats(r2_2)
    mse_1_stats = summary_stats(mse_1)
    mse_2_stats = summary_stats(mse_2)

    print("=== R² Comparison ===")
    print(f"Model 1 R²: Mean={r2_1_stats[0]:.4f}, Median={r2_1_stats[1]:.4f}, Max={r2_1_stats[2]:.4f}, Min={r2_1_stats[3]:.4f}")
    print(f"Model 2 R²: Mean={r2_2_stats[0]:.4f}, Median={r2_2_stats[1]:.4f}, Max={r2_2_stats[2]:.4f}, Min={r2_2_stats[3]:.4f}\n")

    print("=== MSE Comparison ===")
    print(f"Model 1 MSE: Mean={mse_1_stats[0]:.4f}, Median={mse_1_stats[1]:.4f}, Max={mse_1_stats[2]:.4f}, Min={mse_1_stats[3]:.4f}")
    print(f"Model 2 MSE: Mean={mse_2_stats[0]:.4f}, Median={mse_2_stats[1]:.4f}, Max={mse_2_stats[2]:.4f}, Min={mse_2_stats[3]:.4f}\n")

    for t in thresholds:
        count1 = np.sum(r2_1 > t)
        count2 = np.sum(r2_2 > t)
        print(f"Signatures with R² > {t}: Model 1 = {count1}, Model 2 = {count2}")
    print()

    better_r2 = np.sum(r2_1 > r2_2)
    better_mse = np.sum(mse_1 < mse_2)
    print(f"Signatures with better R² in Model 1: {better_r2} / {len(r2_1)}")
    print(f"Signatures with better MSE in Model 1: {better_mse} / {len(mse_1)}")


In [3]:
class PredictExposureWith0Sensitivity:
    def __init__(self, model):
        self.model = model

    def fit(self, X_train, y_bin_train, y_exp_train):
        regressors = []
        for i in range(y_bin_train.shape[1]):       # for each of the 29 binarized label
            idx_train = y_bin_train[:, i] == 1  # if the signature for that sample i 0 do not use it for training the regressor on that signature
            if not np.any(idx_train):
                regressors.append(None)
                continue
            X_train_i = np.hstack([X_train[idx_train], y_bin_train[idx_train, i:i+1]])
            y_train_i = y_exp_train[idx_train, i]

            model_i = clone(self.model)
            model_i.fit(X_train_i, y_train_i)
            regressors.append(model_i)
        return regressors

    def evaluate(self, regressors, X_test, y_bin_test, y_exp_test):
        r2_scores = []
        mse_scores = []
        predictions = []

        for i, model_i in enumerate(regressors):
            y_pred_i = np.zeros(y_bin_test.shape[0])
            idx_test = y_bin_test[:, i] == 1

            if model_i is not None and np.any(idx_test):
                X_test_i = np.hstack([X_test[idx_test], y_bin_test[idx_test, i:i+1]])
                y_pred_i[idx_test] = model_i.predict(X_test_i)

                y_test_i = y_exp_test[idx_test, i]
                r2_scores.append(r2_score(y_test_i, y_pred_i[idx_test]))
                mse_scores.append(mean_squared_error(y_test_i, y_pred_i[idx_test]))
            else:
                r2_scores.append(np.nan)
                mse_scores.append(np.nan)

            predictions.append(y_pred_i)

        return r2_scores, mse_scores, predictions


### Prediction with the LightGBMXT model

In [4]:
# predictor = MultilabelPredictor.load('../models/saved/Predictor-0.03')
# new_base_path = '../models/saved/Predictor-0.03'
# pred_mutation_count = pd.read_csv('../simulations/data2/run_1/trinucleotides_counts_sampling_0.03.csv').iloc[:,1:]
# # aggiorna i path per ogni label
# for label in predictor.labels:
#     predictor.predictors[label] = os.path.join(new_base_path, f'Predictor_{label}')
    
# prediction = predictor.predict(pred_mutation_count)

## Prediction with the GT

### Split train and test Data

train feature data:
- 96 values indicating the mutation count for each type of mutation
- 29 binary labels

train target data:
- 29 real numbers indicating the exposure value 

In [5]:
seed = np.random.randint(1,100000)
np.random.seed(seed=seed)
train_size = 0.9
# Loading the data
signature_prob_distribution = pd.read_csv('../simulations/ground_truth/signatures.csv').iloc[:,1:].values
mutation_count = pd.read_csv('../simulations/data/run_1/trinucleotides_counts_sampling_0.03.csv').iloc[:,1:].values  # N x 96
mutation_count_bin = pd.read_csv('../simulations/ground_truth/bin_exposures.csv').iloc[:,1:].astype(int).values  # N x 29

signature_exposure = pd.read_csv('../simulations/ground_truth/exposures.csv').iloc[:,1:].values  # N x 29
X_train, X_test, y_bin_train, y_bin_test, y_exp_train, y_exp_test = train_test_split(
            mutation_count, mutation_count_bin, signature_exposure, random_state = seed, train_size=train_size,
        )

### Tissues evaluation

In [6]:
from sklearn.preprocessing import LabelEncoder
tissues = pd.read_csv('../simulations/ground_truth/tumor_site.csv').iloc[:,1:-1].values
encoder = LabelEncoder()
tissues_encoded = pd.DataFrame(encoder.fit_transform(tissues))

/home/massimo/Documents/stage/signature_inference/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


###

In [7]:
pes_0 = PredictExposureWith0Sensitivity(LinearRegression())
regressors = pes_0.fit(X_train, y_bin_train, y_exp_train)
r2_0,mse_0,pred_0 = pes_0.evaluate(regressors, X_test, y_bin_test, y_exp_test)

In [8]:
pes_1 = PredictExposureWith0Sensitivity(LinearRegression(positive=True))
regressors = pes_1.fit(X_train, y_bin_train, y_exp_train)
r2_1,mse_1,pred_1 = pes_1.evaluate(regressors, X_test, y_bin_test, y_exp_test)

In [9]:
compare_models(r2_0,mse_0,r2_1, mse_1)

=== R² Comparison ===
Model 1 R²: Mean=0.6463, Median=0.8849, Max=0.9988, Min=-4.5684
Model 2 R²: Mean=0.4079, Median=0.7476, Max=0.9975, Min=-5.7976

=== MSE Comparison ===
Model 1 MSE: Mean=19560969.2817, Median=1680310.5458, Max=469100480.9969, Min=161493.4896
Model 2 MSE: Mean=24382189.9327, Median=3208719.1645, Max=358334546.3090, Min=251301.3691

Signatures with R² > 0.5: Model 1 = 25, Model 2 = 25
Signatures with R² > 0.8: Model 1 = 18, Model 2 = 10

Signatures with better R² in Model 1: 26 / 29
Signatures with better MSE in Model 1: 26 / 29


In [10]:
signature_exposure = pd.read_csv('../simulations/ground_truth/exposures.csv').iloc[:,1:].values  # N x 29
X_train, X_test, y_bin_train, y_bin_test, y_exp_train, y_exp_test = train_test_split(
            mutation_count @ signature_prob_distribution.T, mutation_count_bin, signature_exposure, random_state = seed, train_size=train_size,
        )

In [11]:
pes_2 = PredictExposureWith0Sensitivity(LinearRegression())
regressors = pes_2.fit(X_train, y_bin_train, y_exp_train)
r2_2,mse_2,pred_2 = pes_2.evaluate(regressors, X_test, y_bin_test, y_exp_test)

In [12]:
compare_models(r2_0,mse_0,r2_2, mse_2)

=== R² Comparison ===
Model 1 R²: Mean=0.6463, Median=0.8849, Max=0.9988, Min=-4.5684
Model 2 R²: Mean=0.6214, Median=0.8746, Max=0.9991, Min=-5.4142

=== MSE Comparison ===
Model 1 MSE: Mean=19560969.2817, Median=1680310.5458, Max=469100480.9969, Min=161493.4896
Model 2 MSE: Mean=8936920.6788, Median=1897383.6674, Max=150984820.7525, Min=185371.9605

Signatures with R² > 0.5: Model 1 = 25, Model 2 = 26
Signatures with R² > 0.8: Model 1 = 18, Model 2 = 17

Signatures with better R² in Model 1: 19 / 29
Signatures with better MSE in Model 1: 19 / 29


In [13]:
X_train, X_test, y_bin_train, y_bin_test, y_exp_train, y_exp_test = train_test_split(
            np.hstack([mutation_count @ signature_prob_distribution.T,tissues_encoded]), mutation_count_bin, signature_exposure, random_state = seed, train_size=train_size,
        )

In [14]:
pes_3 = PredictExposureWith0Sensitivity(LinearRegression())
regressors = pes_3.fit(X_train, y_bin_train, y_exp_train)
r2_3,mse_3,pred_3 = pes_3.evaluate(regressors, X_test, y_bin_test, y_exp_test)

In [15]:
compare_models(r2_0,mse_0,r2_3, mse_3)

=== R² Comparison ===
Model 1 R²: Mean=0.6463, Median=0.8849, Max=0.9988, Min=-4.5684
Model 2 R²: Mean=0.6219, Median=0.8746, Max=0.9991, Min=-5.3910

=== MSE Comparison ===
Model 1 MSE: Mean=19560969.2817, Median=1680310.5458, Max=469100480.9969, Min=161493.4896
Model 2 MSE: Mean=8929746.8607, Median=1900230.3240, Max=151261048.5570, Min=185708.2063

Signatures with R² > 0.5: Model 1 = 25, Model 2 = 26
Signatures with R² > 0.8: Model 1 = 18, Model 2 = 17

Signatures with better R² in Model 1: 19 / 29
Signatures with better MSE in Model 1: 19 / 29
